In [9]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import sys
import types
from transformers import modeling_utils

# Monkey patch for older transformers (if needed)
def shard_checkpoint(state_dict, max_shard_size, weights_name):
    return [(state_dict, weights_name)]

if not hasattr(modeling_utils, 'shard_checkpoint'):
    modeling_utils.shard_checkpoint = shard_checkpoint
    print("✓ Monkey patched shard_checkpoint")

# ------------------------------------------------------------
# 1. Load the base model (without adapter)
# ------------------------------------------------------------
base_model_name = "unsloth/Qwen2.5-1.5B"
saved_adapter_path = "outputs/agent-model"

print(f"Loading base model: {base_model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=2048,
    load_in_4bit=True,
    device_map="cuda:0",  # Explicit device
)

# ------------------------------------------------------------
# 2. Load the tokenizer from the saved directory
# ------------------------------------------------------------
print("Loading tokenizer from saved directory...")
saved_tokenizer = AutoTokenizer.from_pretrained(saved_adapter_path)

# ------------------------------------------------------------
# 3. Resize model embeddings to match the saved tokenizer
# ------------------------------------------------------------
print(f"Resizing embeddings from {len(tokenizer)} to {len(saved_tokenizer)}...")
model.resize_token_embeddings(len(saved_tokenizer))
tokenizer = saved_tokenizer

# ------------------------------------------------------------
# 4. Load the LoRA adapter (before moving to ensure device)
# ------------------------------------------------------------
print("Loading adapter...")
model.load_adapter(saved_adapter_path)

# ------------------------------------------------------------
# 5. Explicitly move everything to GPU
# ------------------------------------------------------------
print("Moving all parameters to GPU...")
model = model.to('cuda:0')
# Ensure embedding and lm_head are on GPU (redundant but safe)
if hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
    model.model.embed_tokens = model.model.embed_tokens.to('cuda:0')
if hasattr(model, 'lm_head'):
    model.lm_head = model.lm_head.to('cuda:0')

# ------------------------------------------------------------
# 6. Prepare model for inference
# ------------------------------------------------------------
model.eval()
FastLanguageModel.for_inference(model)

# ------------------------------------------------------------
# 7. Generation function
# ------------------------------------------------------------
def generate_response(prompt: str, max_new_tokens=128, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to('cuda:0') for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    response_ids = outputs[0][input_len:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)
    return response.strip()

# ------------------------------------------------------------
# 8. Interactive loop
# ------------------------------------------------------------
print("Model ready! Enter your prompts (type 'quit' to exit).")
while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ["quit", "exit"]:
        break
    response = generate_response(user_input)
    print(f"Assistant: {response}")

Loading base model: unsloth/Qwen2.5-1.5B...==((====))==  Unsloth 2026.3.10: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 2060. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.Loading tokenizer from saved directory...Resizing embeddings from 151666 to 151672...
Loading adapter...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Qwen2ForCausalLM LOAD REPORT from: outputs/agent-model
Key                             | Status     |  | 
--------------------------------+------------+--+-
base_model.model.lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


User:  Help me with: compare AI trends in Bangalore


Moving all parameters to GPU...Model ready! Enter your prompts (type 'quit' to exit).Assistant: Thought: I need to use file_reader to gather information about compare AI trends in Bangalore.

Action:
ภู
จัยfile_readerحَ
บริษัท
{
  "file_path": "/data/compare_AI_trends_in_Bangalore_12.txt",
  "operation": "summarize"
}
อิส
แข็งแรง
สมเด็
郎
ยิง
จุ
K
 โดยมี
สมบู
ขวั
لَّ
อีเมล
อุดม
สมัครสมาชิก
ชั่วโมง
เกือบ
สถิติ
يّ
練
พิเศษ
ไข่
ได้ว่า
เรื่
โต๊
תּ
ข่าวสาร
 วัน
มั่น
เทีย
ตัวอย่าง
ปฏิ
มั้
สัญญาณ

KeyboardInterrupt: Interrupted by user